# Extended experiments (3-6): sensitivity, AUC battery, bottleneck, fragility

Run **after** `run_experiments.ipynb` has produced `OUT_DIR/obstruction.csv`
for this model (experiments 4 uses that CSV; 3, 5, 6 are standalone).
Cohort additionally needs `target_text` per record for the bottleneck run.

In [ ]:
MODEL_NAME  = "meta-llama/Llama-3.2-3B"
COHORT_PATH = "cohorts/llama32_3b_parametric.jsonl"
OUT_DIR     = "results/notebook_run"          # must contain obstruction.csv
DEVICE      = "cuda"

import os, json, torch
torch.set_grad_enabled(False)
from transformers import AutoModelForCausalLM, AutoTokenizer
from frozen_cache import Weights
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32,
    attn_implementation="eager").to(DEVICE).eval()
W = Weights(model)
records = [json.loads(l) for l in open(COHORT_PATH)]
print(f"{MODEL_NAME}: L={W.L}; {len(records)} records")

## Experiment 3 — threshold sensitivity
Verdict stability over pin_L x c_plus-quantile. The claim: the decomposition
is a property of the computation, not the calibration.

In [ ]:
from run_sensitivity import sweep
import pandas as pd
sens = pd.DataFrame(sweep(model, tok, W, records, DEVICE))
sens.to_csv(f"{OUT_DIR}/sensitivity.csv", index=False)
print(f"min agreement with reference config: {sens.agree_with_ref.min():.2f}")
sens.round(3)

## Experiment 4 — three-tier AUC battery
Black-box vs paper internals vs affine features. Uses experiment 1's CSV.

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "run_auc_battery.py",
    "--model", MODEL_NAME, "--cohort", COHORT_PATH,
    "--obcsv", f"{OUT_DIR}/obstruction.csv", "--out", OUT_DIR,
    "--device", DEVICE], capture_output=True, text=True)
print(r.stdout or r.stderr)

## Experiment 5 — first-token bottleneck by verdict
Needs `target_text` in the cohort. Prediction: div@0 + back-on-rails
concentrate in selection/transport, not source.

In [ ]:
r = subprocess.run([sys.executable, "run_bottleneck.py",
    "--model", MODEL_NAME, "--cohort", COHORT_PATH, "--out", OUT_DIR,
    "--device", DEVICE], capture_output=True, text=True)
print(r.stdout[-3000:] or r.stderr[-3000:])

## Experiment 6 — spectral fragility (normal-equations gap)
sigma_min of the pinned dynamics operator. Prediction: transport failures and
paraphrase-unstable examples sit at small gaps. Costly (~power_iters x
cg_iters frozen passes per example) — run on a subset first.

In [ ]:
from frozen_cache import build_cache
from obstruction import fragility
import pandas as pd

SUBSET = 40   # examples; raise once timing is known
rows = []
for i, r in enumerate(records[:SUBSET]):
    ids = tok(r["prompt"], return_tensors="pt").input_ids[0].to(DEVICE)
    C = build_cache(model, W, ids)
    S = list(range(r["source_span"][0], r["source_span"][1] + 1))
    sig = fragility(W, C, S)
    rows.append(dict(idx=i, verdict=r["verdict"], sigma_min=sig))
    print(f"[{i:3d}] {r['verdict']:<9s} sigma_min={sig:.4f}")
fdf = pd.DataFrame(rows)
fdf.to_csv(f"{OUT_DIR}/fragility.csv", index=False)
fdf.groupby("verdict").sigma_min.median()

## Cross-model equivalence bound
After all models have run experiment 1:

In [ ]:
# !python stats_utils.py results/*/obstruction.csv

In [ ]:
# extended smoke test in this kernel's env:
# import subprocess, sys
# print(subprocess.run([sys.executable, "smoke_test_extended.py"],
#                      capture_output=True, text=True).stdout)